# 02 - Heaps analysis

Presentation-only notebook. Generate or refresh results with `python -m yelptext analyze heaps --config configs/full.toml`. Randomization, tokenization, checkpointing, fitting, sensitivity analysis, and provenance checks run in the registered command, not here.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / 'pyproject.toml').is_file():
            return path
    raise RuntimeError('Run inside the project repository')

ROOT = find_root()
ARTIFACTS = ROOT / 'artifacts' / 'heaps'
summary = json.loads((ARTIFACTS / 'heaps_summary.json').read_text(encoding='utf-8'))
if summary.get('status') != 'COMPLETED' or 'main_fit' not in summary:
    raise RuntimeError('Run: python -m yelptext analyze heaps --config configs/full.toml')
checkpoints = pd.read_csv(ARTIFACTS / 'main_checkpoints.csv')
sensitivity = pd.read_csv(ARTIFACTS / 'sensitivity_parameters.csv')


## Corpus, model, and endpoint validation

In [ ]:
fit = summary['main_fit']
prediction = summary['prediction']
pd.DataFrame([{
    'reviews': summary['reviews'],
    'tokens_N': summary['total_tokens'],
    'types_V': summary['vocabulary_size'],
    'k': fit['k'],
    'beta': fit['beta'],
    'R2_log_space': fit['r_squared_log_space'],
    'prediction_APE_percent': prediction['absolute_percentage_error'],
    'endpoint_matches_vocabulary': summary['endpoint_matches_vocabulary'],
}])


In [ ]:
display(checkpoints)
display(Image(filename=str(ARTIFACTS / 'heaps_loglog.png')))
display(Image(filename=str(ARTIFACTS / 'heaps_prediction.png')))


## Checkpoint and fitted-interval sensitivity

In [ ]:
sensitivity.style.format({
    'k': '{:.4f}', 'beta': '{:.5f}', 'r_squared_log_space': '{:.6f}',
    'delta_k_percent': '{:.2f}', 'delta_beta': '{:.5f}',
    'reference_prediction_delta_percent': '{:.3f}',
})


The deterministic seed controls review accumulation order. Checkpoints are nested observations from one corpus and are therefore correlated; the 10/30/50% extrapolation uses the observed final token count and is not an independent-corpus evaluation.